# BP1 Executive Rollup Report — Customer Intent Classification
**Customer360 Navigator Enterprise Suite — Customer Intent Classification**

## Purpose
Produces the single, comprehensive executive-rollup deliverable for BP1: a world-class interactive
HTML dashboard (slicers, filters, animated KPIs, varied chart types, SMART suggestions), a Word
report, an Excel workbook (multiple sheets), and a PowerPoint deck — all built entirely from BP1's
real Gates 1-6 artifacts, which must already be real-run confirmed on this machine.

## What this notebook does, concretely
This notebook trains and evaluates nothing itself — every number, table, and chart it produces was
already computed and recorded by Gates 1-6's own real runs. It performs four real actions:
1. **Reads every real Gate 1-6 artifact live** via `src/reporting/bp1_rollup_helpers.load_all_gate_artifacts()`
   — the shared config YAML plus every JSON/CSV artifact file each gate wrote.
2. **Assembles real KPIs, data-grounded SMART suggestions, best/worst-performing intents, the top
   real confused-intent pairs, and full gate-by-gate detail for Gate 1 (Business Understanding &
   Policy) and Gate 6 (Governance & Known Limitations)** — the two gates whose real output is not
   otherwise represented by a chart. Every figure here is read live or computed live from Gates
   1-6, never freeform GenAI text and never an assumption-based estimate. **There is no
   financial-impact or illustrative-projection section anywhere in this report** — BP1's source
   data (BANKING77 + the CFPB structured extract) contains no real cost or volume figure of its
   own, and only original notebook output results are reported here, per standing instruction.
3. **Renders five static figures once** (WARP: model benchmark bar, SHAP top-features bar, taxonomy
   pie, confidence-distribution histogram, confusion heatmap) and reuses the same PNG bytes across
   the Word and PowerPoint exports.
4. **Writes four deliverables** to `reports/bp1_customer_intent_classification/executive_rollup/`
   inside this real project folder:
   - `bp1_executive_rollup_dashboard.html` — self-contained interactive dashboard (Plotly.js via
     CDN, client-side slicers/filters, animated KPI cards — no new Python dependency)
   - `bp1_executive_rollup_report.docx` — Word report with charts, tables, and SMART suggestions,
     covering all six gates
   - `bp1_executive_rollup_workbook.xlsx` — Excel workbook, 11 sheets, one per real gate-output area
   - `bp1_executive_rollup_deck.pptx` — PowerPoint deck, 10 slides

## Standing rules this notebook follows
- **Zero-fabrication, no assumption-based content, only original notebook output results**: every
  KPI, table, and chart in every deliverable is read live from Gates 1-6's own already-recorded
  real artifacts, or computed live from them by a documented formula over those real values. There
  is no illustrative, estimated, or assumption-based content anywhere in this report.
- **Comprehensive coverage**: all six BP1 gates' real recorded output is represented somewhere in
  every deliverable — including Gate 1 (Business Understanding & Policy) and Gate 6 (Governance &
  Known Limitations), which a chart-only rollup would otherwise leave out entirely.
- **HYPER**: all data-loading, KPI assembly, SMART-suggestion generation, figure builders, and
  DOCX/XLSX/PPTX writers live in one shared module (`src/reporting/bp1_rollup_helpers.py`),
  imported once here — this notebook itself is a thin orchestrator, not a second copy of that logic.
- **WARP**: `configure_performance()` is called first, before any heavy/BLAS-backed import; every
  matplotlib figure is rendered exactly once and reused as PNG bytes.
- **Idempotent**: re-running overwrites this report's four output files and the executive-rollup
  manifest JSON in place.
- **All outputs are written under this real project folder** —
  `reports/bp1_customer_intent_classification/executive_rollup/` — never anywhere else, per
  standing instruction.
- **No execution-based verification of this notebook's own code was performed by Claude** — not
  even against synthetic fixtures — per the user's explicit instruction. This notebook was checked
  only by static means (valid notebook JSON structure, `ast.parse`, `compile()`, `pyflakes`) before
  being handed to you. You are the first and only one to actually run it.

## Outputs (idempotent overwrite-in-place)
- `reports/bp1_customer_intent_classification/executive_rollup/bp1_executive_rollup_dashboard.html`
- `reports/bp1_customer_intent_classification/executive_rollup/bp1_executive_rollup_report.docx`
- `reports/bp1_customer_intent_classification/executive_rollup/bp1_executive_rollup_workbook.xlsx`
- `reports/bp1_customer_intent_classification/executive_rollup/bp1_executive_rollup_deck.pptx`
- `notebooks/bp1_customer_intent_classification/artifacts/executive_rollup_manifest.json`

## Prerequisites
BP1 Gates 1-6 must all have been real-run at least once (all six are real-run confirmed as of this
writing). This notebook also requires `python-docx`, `openpyxl`, and `python-pptx` to be installed
in whichever environment runs it — it checks for them explicitly and raises a clear error naming
any that are missing, rather than failing deep inside an export function.

## If a structural check below fails
It raises `AssertionError` naming the failing check. Do not silence it.


In [ ]:
# ============================================================
# SECTION 1: Project root resolution (PROJECT_STRUCTURE_LOCKED.md rule #3)
# ============================================================
import os
import sys
from pathlib import Path


def _find_project_root(marker_filename: str = "PROJECT_STRUCTURE_LOCKED.md") -> Path:
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        candidate = Path(env_override)
        if (candidate / marker_filename).exists():
            return candidate
        raise RuntimeError(
            f"C360_PROJECT_ROOT is set to {candidate} but {marker_filename} was not found there. "
            "Fix the environment variable rather than removing this check."
        )

    start = Path.cwd()
    current = start
    for _ in range(8):
        if (current / marker_filename).exists():
            return current
        if current.parent == current:
            break
        current = current.parent

    for depth_root, dirnames, filenames in os.walk(start):
        rel_depth = len(Path(depth_root).relative_to(start).parts)
        if rel_depth > 3:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        if marker_filename in filenames:
            return Path(depth_root)

    raise RuntimeError(
        "Could not resolve PROJECT_ROOT. Set the C360_PROJECT_ROOT environment variable to the "
        "Customer360_Navigator_Enterprise_Suite folder, or run this notebook from inside the project tree "
        "(expected at notebooks/bp1_customer_intent_classification/)."
    )


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
CONFIGS_DIR = PROJECT_ROOT / "configs"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "bp1_customer_intent_classification" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR = PROJECT_ROOT / "reports" / "bp1_customer_intent_classification"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
ROLLUP_DIR = REPORTS_DIR / "executive_rollup"
ROLLUP_DIR.mkdir(parents=True, exist_ok=True)
print(f"[OK] Project root resolved: {PROJECT_ROOT.name}")
print(f"[OK] Executive-rollup output folder ready: {ROLLUP_DIR.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 2: WARP - configure_performance() FIRST, before any heavy/BLAS-backed import
# ============================================================
from utils.performance_setup import configure_performance  # noqa: E402

WARP_SUMMARY = configure_performance(project_root=PROJECT_ROOT, verbose=True)

# ============================================================
# SECTION 3: Load the shared reporting helper module (HYPER) and all real Gate 1-6 artifacts
# ============================================================
import reporting.bp1_rollup_helpers as rollup  # noqa: E402

_missing_reporting_deps = []
for _mod in ("docx", "openpyxl", "pptx"):
    try:
        __import__(_mod)
    except ImportError:
        _missing_reporting_deps.append(_mod)
if _missing_reporting_deps:
    raise ImportError(
        "Executive-rollup report requires the following packages, missing from this environment: "
        f"{_missing_reporting_deps}. Install them (pip install python-docx openpyxl python-pptx) in "
        "the same environment this notebook runs in, then re-run."
    )
print("[OK] docx / openpyxl / pptx reporting dependencies available.")

BUNDLE = rollup.load_all_gate_artifacts(PROJECT_ROOT)
print(f"[OK] Loaded real Gate 1-6 artifacts. Champion model: {BUNDLE['model_inventory']['model_name']}")

# ============================================================
# SECTION 4: KPI bundle, Gate 1 + Gate 6 real-output detail, SMART suggestions, best/worst intents,
# top confused pairs - all real, computed live from Gates 1-6's own artifacts (HYPER: one
# computation each, reused by every export). No financial-impact / assumption-based section exists
# anywhere in this notebook, per standing instruction - only original notebook output results.
# ============================================================
KPIS = rollup.build_kpi_bundle(BUNDLE)
GATE1 = rollup.build_gate1_summary(BUNDLE)
GATE6_DETAIL = rollup.build_gate6_governance_detail(BUNDLE)
SUGGESTIONS = rollup.build_smart_suggestions(BUNDLE)
BEST_INTENTS, WORST_INTENTS = rollup.worst_best_intents(BUNDLE["gate3_classification_report"], n=10)
CONFUSED_PAIRS = rollup.top_confused_pairs(BUNDLE["gate3_confusion_df"], n=50)
print(
    f"[OK] KPI bundle assembled ({len(KPIS)} keys). Gate 1 policy detail ({len(GATE1)} fields) and "
    f"Gate 6 governance detail ({len(GATE6_DETAIL)} fields) assembled. {len(SUGGESTIONS)} SMART "
    f"suggestions generated (all data-grounded, never freeform GenAI text). Best/worst intents: "
    f"{len(BEST_INTENTS)}/{len(WORST_INTENTS)} rows. Top confused pairs: {len(CONFUSED_PAIRS)} rows."
)

# ============================================================
# SECTION 5: Render every static figure ONCE (WARP) - reused as raw PNG bytes across the DOCX and
# PPTX exports below, and never touched as a live pyplot Figure object more than once.
# ============================================================
FIGURES = {
    "model_benchmark": rollup.fig_model_comparison_bar(BUNDLE["gate3_cv_df"], KPIS["champion_model"]),
    "shap_top": rollup.fig_shap_top_features(BUNDLE["gate4_shap_df"]),
    "taxonomy_pie": rollup.fig_taxonomy_bucket_pie(BUNDLE["gate2_coverage_df"]),
    "confidence_dist": rollup.fig_confidence_distribution(BUNDLE["gate5_decision_df"]),
    "confusion_heatmap": rollup.fig_confusion_heatmap_top(BUNDLE["gate3_confusion_df"], CONFUSED_PAIRS),
}
for _name, _png_bytes in FIGURES.items():
    print(f"[OK] Figure rendered: {_name} ({len(_png_bytes):,} bytes PNG)")

# ============================================================
# SECTION 6: Interactive HTML dashboard (Plotly.js via CDN, client-side slicers/filters - no new
# Python dependency). All data embedded as one JSON payload; template placeholders substituted via
# plain string .replace(), never f-strings, to avoid brace-escaping conflicts with the template's
# own CSS/JS.
# ============================================================
import json
import numpy as np


def _json_default(obj):
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.bool_):
        return bool(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, Path):
        return str(obj)
    raise TypeError(f"Object of type {type(obj)} is not JSON serializable")


_gate2_df = BUNDLE["gate2_coverage_df"]
_gate3_cv_ok = BUNDLE["gate3_cv_df"][BUNDLE["gate3_cv_df"]["status"] == "OK"]

ROLLUP_DATA = {
    "palette": rollup.PALETTE,
    "categorical_sequence": rollup.CATEGORICAL_SEQUENCE,
    "kpis": KPIS,
    "gate1": GATE1,
    "gate6_detail": GATE6_DETAIL,
    "taxonomy": [
        {"bucket": r["common_taxonomy_bucket"], "cfpb_count": int(r["cfpb_row_count"])}
        for _, r in _gate2_df.iterrows()
    ],
    "model_benchmark": [
        {
            "model": r["model"],
            "mean_f1_macro": float(r["mean_f1_macro"]),
            "std_f1_macro": float(r["std_f1_macro"]),
            "is_champion": bool(r["model"] == KPIS["champion_model"]),
        }
        for _, r in _gate3_cv_ok.iterrows()
    ],
    "shap_top10": BUNDLE["gate4_shap_df"].head(10)[["feature", "mean_abs_shap"]].to_dict("records"),
    "confusion_matrix": {
        "matrix": BUNDLE["gate3_confusion_df"].values.tolist(),
        "labels": list(BUNDLE["gate3_confusion_df"].index),
    },
    "confused_pairs": CONFUSED_PAIRS.to_dict("records"),
    "decision_records": BUNDLE["gate5_decision_df"][["confidence_top1", "correct"]].to_dict("records"),
    "best_intents": BEST_INTENTS.to_dict("records"),
    "worst_intents": WORST_INTENTS.to_dict("records"),
    "suggestions": SUGGESTIONS,
}

_data_json = json.dumps(ROLLUP_DATA, default=_json_default)
_data_json_safe = _data_json.replace("</", "<\\/")  # never let embedded text close the script tag early

_pytest_counts = KPIS["pytest_counts"]
_pytest_summary = BUNDLE["gate6_summary"].get(
    "pytest_summary_line", f"{_pytest_counts['passed']} passed / {_pytest_counts['failed']} failed"
)
_syntax_n_pass = BUNDLE["gate6_summary"]["notebook_syntax_check_n_passed"]
_syntax_n_fail = BUNDLE["gate6_summary"]["notebook_syntax_check_n_failed"]
_syntax_summary = f"{_syntax_n_pass}/{_syntax_n_pass + _syntax_n_fail} passed"

TEMPLATE_PATH = PROJECT_ROOT / "src" / "reporting" / "templates" / "bp1_dashboard_template.html"
_html_text = TEMPLATE_PATH.read_text(encoding="utf-8")
_html_text = _html_text.replace("__GENERATED_AT__", KPIS["generated_at_utc"])
_html_text = _html_text.replace("__CHAMPION_MODEL__", KPIS["champion_model"])
_html_text = _html_text.replace("__PYTEST_SUMMARY__", _pytest_summary)
_html_text = _html_text.replace("__SYNTAX_SUMMARY__", _syntax_summary)
_html_text = _html_text.replace("__ROLLUP_DATA_JSON__", _data_json_safe)

DASHBOARD_HTML_PATH = ROLLUP_DIR / "bp1_executive_rollup_dashboard.html"
DASHBOARD_HTML_PATH.write_text(_html_text, encoding="utf-8")
print(f"[SAVED] {DASHBOARD_HTML_PATH.relative_to(PROJECT_ROOT)} ({DASHBOARD_HTML_PATH.stat().st_size:,} bytes)")

# ============================================================
# SECTION 7: Word document export (python-docx) - charts, tables, SMART suggestions, and full
# Gate 1 / Gate 6 real-output detail (docx skill: US Letter page size set explicitly, Calibri
# professional font throughout). No financial-impact section exists in this report.
# ============================================================
DOCX_PATH = ROLLUP_DIR / "bp1_executive_rollup_report.docx"
rollup.write_docx_report(
    BUNDLE, KPIS, SUGGESTIONS, FIGURES, BEST_INTENTS, WORST_INTENTS, CONFUSED_PAIRS, DOCX_PATH,
)
print(f"[SAVED] {DOCX_PATH.relative_to(PROJECT_ROOT)} ({DOCX_PATH.stat().st_size:,} bytes)")

# ============================================================
# SECTION 8: Excel workbook export (openpyxl) - 11 sheets, one per real gate-output area
# (Gate 1 policy, Gate 3 benchmark/statistics/confusion, Gate 4 SHAP, Gate 5 decision layer,
# Gate 2 taxonomy, Gate 6 governance, SMART suggestions). No financial-impact sheet.
# ============================================================
XLSX_PATH = ROLLUP_DIR / "bp1_executive_rollup_workbook.xlsx"
rollup.write_xlsx_workbook(
    BUNDLE, KPIS, SUGGESTIONS, BEST_INTENTS, WORST_INTENTS, CONFUSED_PAIRS, XLSX_PATH,
)
print(f"[SAVED] {XLSX_PATH.relative_to(PROJECT_ROOT)} ({XLSX_PATH.stat().st_size:,} bytes)")

# ============================================================
# SECTION 9: PowerPoint deck export (python-pptx) - explicit slide size, RGBColor objects (never a
# literal '#' prefix or 8-digit hex), figures rendered once and reused (WARP). No financial slide.
# ============================================================
PPTX_PATH = ROLLUP_DIR / "bp1_executive_rollup_deck.pptx"
rollup.write_pptx_deck(BUNDLE, KPIS, SUGGESTIONS, FIGURES, PPTX_PATH)
print(f"[SAVED] {PPTX_PATH.relative_to(PROJECT_ROOT)} ({PPTX_PATH.stat().st_size:,} bytes)")

# ============================================================
# SECTION 10: Executive-rollup manifest (audit trail - which real Gate 1-6 artifacts fed this
# report, and what was generated, when).
# ============================================================
from datetime import datetime, timezone  # noqa: E402

MANIFEST = {
    "report": "bp1_customer_intent_classification_executive_rollup",
    "champion_model": KPIS["champion_model"],
    "source_artifacts_dir": str(BUNDLE["artifacts_dir"].relative_to(PROJECT_ROOT)),
    "source_artifact_files": sorted(p.name for p in BUNDLE["artifacts_dir"].glob("*") if p.is_file()),
    "outputs": {
        "dashboard_html": str(DASHBOARD_HTML_PATH.relative_to(PROJECT_ROOT)),
        "report_docx": str(DOCX_PATH.relative_to(PROJECT_ROOT)),
        "workbook_xlsx": str(XLSX_PATH.relative_to(PROJECT_ROOT)),
        "deck_pptx": str(PPTX_PATH.relative_to(PROJECT_ROOT)),
    },
    "output_sizes_bytes": {
        "dashboard_html": DASHBOARD_HTML_PATH.stat().st_size,
        "report_docx": DOCX_PATH.stat().st_size,
        "workbook_xlsx": XLSX_PATH.stat().st_size,
        "deck_pptx": PPTX_PATH.stat().st_size,
    },
    "contains_financial_impact_section": False,
    "contains_assumption_based_content": False,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
}
MANIFEST_PATH = ARTIFACTS_DIR / "executive_rollup_manifest.json"
with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(MANIFEST, f, indent=2)
print(f"[SAVED] {MANIFEST_PATH.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 11: Structural integrity checks - raise AssertionError, never silently pass.
# ============================================================
from docx import Document as _DocxDocument  # noqa: E402
from openpyxl import load_workbook as _load_workbook  # noqa: E402
from pptx import Presentation as _PptxPresentation  # noqa: E402

_docx_reopen_ok = _DocxDocument(str(DOCX_PATH)) is not None
_xlsx_reopen_ok = _load_workbook(str(XLSX_PATH)) is not None
_pptx_reopen_ok = _PptxPresentation(str(PPTX_PATH)) is not None

_saved_html = DASHBOARD_HTML_PATH.read_text(encoding="utf-8")
_embedded_json_text = _saved_html.split(
    '<script id="rollup-data" type="application/json">', 1
)[1].split("</script>", 1)[0].replace("<\\/", "</")
_embedded_json = json.loads(_embedded_json_text)

_xlsx_wb_check = _load_workbook(str(XLSX_PATH))

checks = {
    "dashboard_html_written_and_nonempty": DASHBOARD_HTML_PATH.exists() and DASHBOARD_HTML_PATH.stat().st_size > 10_000,
    "dashboard_html_embedded_json_parses": isinstance(_embedded_json, dict) and "kpis" in _embedded_json,
    "dashboard_html_includes_gate1_and_gate6": "gate1" in _embedded_json and "gate6_detail" in _embedded_json,
    "dashboard_html_no_financial_content": "financial" not in _embedded_json and "Financial Impact" not in _saved_html,
    "dashboard_html_no_unresolved_placeholders": all(
        token not in _saved_html
        for token in ("__GENERATED_AT__", "__CHAMPION_MODEL__", "__PYTEST_SUMMARY__", "__SYNTAX_SUMMARY__", "__ROLLUP_DATA_JSON__")
    ),
    "docx_written_and_nonempty": DOCX_PATH.exists() and DOCX_PATH.stat().st_size > 10_000,
    "xlsx_written_and_nonempty": XLSX_PATH.exists() and XLSX_PATH.stat().st_size > 5_000,
    "xlsx_has_gate1_and_gate6_sheets": (
        "02_Gate1_Business_Policy" in _xlsx_wb_check.sheetnames
        and "09_Gate6_Governance_Limitations" in _xlsx_wb_check.sheetnames
    ),
    "xlsx_no_financial_sheet": not any("Financial" in name for name in _xlsx_wb_check.sheetnames),
    "pptx_written_and_nonempty": PPTX_PATH.exists() and PPTX_PATH.stat().st_size > 10_000,
    "manifest_written": MANIFEST_PATH.exists(),
    "docx_reopens_cleanly": _docx_reopen_ok,
    "xlsx_reopens_cleanly": _xlsx_reopen_ok,
    "pptx_reopens_cleanly": _pptx_reopen_ok,
    "smart_suggestions_all_grounded": len(SUGGESTIONS) >= 3,
    "outputs_written_under_project_reports_folder": all(
        str(p).startswith(str(REPORTS_DIR)) for p in [DASHBOARD_HTML_PATH, DOCX_PATH, XLSX_PATH, PPTX_PATH]
    ),
}

print("\n=== INTEGRITY CHECKS ===")
for _name, _passed in checks.items():
    _status = "[PASS]" if _passed else "[FAIL]"
    print(f"{_status} {_name}")
    assert _passed, f"[CHECK FAILED] {_name}"

print(
    f"\n[ALL CHECKS PASSED] BP1 executive-rollup report complete. "
    f"Dashboard: {DASHBOARD_HTML_PATH.name} | Report: {DOCX_PATH.name} | "
    f"Workbook: {XLSX_PATH.name} | Deck: {PPTX_PATH.name}. "
    f"All outputs written under {ROLLUP_DIR.relative_to(PROJECT_ROOT)}. "
    "Every figure in every deliverable is a real, original BP1 Gate 1-6 notebook output - "
    "there is no financial-impact section and no assumption-based content anywhere in this report."
)
